# Tasty Bytes: 全テーブルの作成とデータ取り込み

Tasty Bytes の各教材が S3 から読み込むテーブルを、1 つのデータベース `tasty_bytes_all` にまとめて作成し、データを取り込みます。

- テーブル定義（列・型）と `COPY INTO` のオプションは、各教材のセットアップ SQL のとおりです。出典は各セクションに記載しています。
- 複数の教材が同じテーブルを別のソースから読み込んでいる場合は、1 つだけを採用しています（例: 中核の POS テーブルは `s3://sfquickstarts/frostbyte_tastybytes/` のみ）。
- 採用しなかったソースや、S3 から取り込めないデータ（Marketplace、手動アップロードのメディア）は、最後のセクションにまとめています。

詳しい解説はリポジトリの `wiki/data-model/` を、テーブル同士の関係は `wiki/data-model/er-diagram.md`（ER 図）を参照してください。

**実行方法**: Snowsight で新規 Notebook を作成してこのファイルをインポートし、上から順に実行します。`SYSADMIN` ロールでデータベースとウェアハウスを作成できる必要があります。

## 1. データベース・スキーマ・ウェアハウス

スキーマ名は元の教材にそろえています。

In [ ]:
USE ROLE sysadmin;

CREATE OR REPLACE DATABASE tasty_bytes_all;

CREATE OR REPLACE SCHEMA tasty_bytes_all.raw_pos;
CREATE OR REPLACE SCHEMA tasty_bytes_all.raw_customer;
CREATE OR REPLACE SCHEMA tasty_bytes_all.raw_support;
CREATE OR REPLACE SCHEMA tasty_bytes_all.raw_supply_chain;
CREATE OR REPLACE SCHEMA tasty_bytes_all.raw_safegraph;
CREATE OR REPLACE SCHEMA tasty_bytes_all.raw_doc;
CREATE OR REPLACE SCHEMA tasty_bytes_all.harmonized;
CREATE OR REPLACE SCHEMA tasty_bytes_all.analytics;
CREATE OR REPLACE SCHEMA tasty_bytes_all.app;

CREATE OR REPLACE WAREHOUSE tasty_bytes_all_wh
    WAREHOUSE_SIZE = 'large'
    WAREHOUSE_TYPE = 'standard'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
COMMENT = 'warehouse for loading all tasty bytes source tables';

USE WAREHOUSE tasty_bytes_all_wh;

## 2. ファイル形式とステージ

教材ごとに CSV の指定が違うため、元の指定をそのまま再現した形式を 3 つ作ります。

| ファイル形式 | 指定 | 元の教材 |
| --- | --- | --- |
| `csv_ff` | `type = 'csv'` のみ | Zero to Snowflake v2、顧客レビュー分析、カスタマーエクスペリエンスアプリ、Snowpark 101、React Native アプリ |
| `csv_header_ff` | ヘッダー 1 行を読み飛ばし、`"` 囲みに対応 | 価格最適化（`tb_po_setup.sql`） |
| `csv_header_auto_ff` | ヘッダー 1 行を読み飛ばし、圧縮を自動判定 | ML 予測・異常検知 |

外部ステージは、S3 バケットごとに 1 つ作ります。

In [ ]:
CREATE OR REPLACE FILE FORMAT tasty_bytes_all.public.csv_ff
type = 'csv';

-- tb_po_setup.sql (price optimization)
CREATE OR REPLACE FILE FORMAT tasty_bytes_all.public.csv_header_ff
type = 'CSV'
field_delimiter = ','
record_delimiter = '\n'
field_optionally_enclosed_by = '"'
skip_header = 1;

-- ml-forecasting-ad
CREATE OR REPLACE FILE FORMAT tasty_bytes_all.public.csv_header_auto_ff
    type = 'csv'
    SKIP_HEADER = 1,
    COMPRESSION = AUTO;

CREATE OR REPLACE STAGE tasty_bytes_all.public.frostbyte_s3
COMMENT = 'Quickstarts S3 Stage Connection'
url = 's3://sfquickstarts/frostbyte_tastybytes/'
file_format = tasty_bytes_all.public.csv_ff;

CREATE OR REPLACE STAGE tasty_bytes_all.public.voc_s3
COMMENT = 'Truck Reviews Stage'
url = 's3://sfquickstarts/tastybytes-voc/'
file_format = tasty_bytes_all.public.csv_ff;

CREATE OR REPLACE STAGE tasty_bytes_all.public.ece_s3
COMMENT = 'Enhancing Customer Experience Stage'
url = 's3://sfquickstarts/tastybytes-ece/'
file_format = tasty_bytes_all.public.csv_ff;

## 3. 中核テーブル（`raw_pos` / `raw_customer`）

出典: `sfguide-getting-started-from-zero-to-snowflake/scripts/setup.sql`（v2）。ソース: `s3://sfquickstarts/frostbyte_tastybytes/raw_pos/*`, `raw_customer/customer_loyalty/`

同じテーブルを読み込むほかのソース（`tasty-bytes-builder-education/`、Parquet の `data-engineering-with-snowpark-python/` など）は採用していません。

In [ ]:
CREATE OR REPLACE TABLE tasty_bytes_all.raw_pos.country
(
    country_id NUMBER(18,0),
    country VARCHAR(16777216),
    iso_currency VARCHAR(3),
    iso_country VARCHAR(2),
    city_id NUMBER(19,0),
    city VARCHAR(16777216),
    city_population VARCHAR(16777216)
);

CREATE OR REPLACE TABLE tasty_bytes_all.raw_pos.franchise
(
    franchise_id NUMBER(38,0),
    first_name VARCHAR(16777216),
    last_name VARCHAR(16777216),
    city VARCHAR(16777216),
    country VARCHAR(16777216),
    e_mail VARCHAR(16777216),
    phone_number VARCHAR(16777216)
);

CREATE OR REPLACE TABLE tasty_bytes_all.raw_pos.location
(
    location_id NUMBER(19,0),
    placekey VARCHAR(16777216),
    location VARCHAR(16777216),
    city VARCHAR(16777216),
    region VARCHAR(16777216),
    iso_country_code VARCHAR(16777216),
    country VARCHAR(16777216)
);

CREATE OR REPLACE TABLE tasty_bytes_all.raw_pos.menu
(
    menu_id NUMBER(19,0),
    menu_type_id NUMBER(38,0),
    menu_type VARCHAR(16777216),
    truck_brand_name VARCHAR(16777216),
    menu_item_id NUMBER(38,0),
    menu_item_name VARCHAR(16777216),
    item_category VARCHAR(16777216),
    item_subcategory VARCHAR(16777216),
    cost_of_goods_usd NUMBER(38,4),
    sale_price_usd NUMBER(38,4),
    menu_item_health_metrics_obj VARIANT
);

CREATE OR REPLACE TABLE tasty_bytes_all.raw_pos.truck
(
    truck_id NUMBER(38,0),
    menu_type_id NUMBER(38,0),
    primary_city VARCHAR(16777216),
    region VARCHAR(16777216),
    iso_region VARCHAR(16777216),
    country VARCHAR(16777216),
    iso_country_code VARCHAR(16777216),
    franchise_flag NUMBER(38,0),
    year NUMBER(38,0),
    make VARCHAR(16777216),
    model VARCHAR(16777216),
    ev_flag NUMBER(38,0),
    franchise_id NUMBER(38,0),
    truck_opening_date DATE
);

CREATE OR REPLACE TABLE tasty_bytes_all.raw_pos.order_header
(
    order_id NUMBER(38,0),
    truck_id NUMBER(38,0),
    location_id FLOAT,
    customer_id NUMBER(38,0),
    discount_id VARCHAR(16777216),
    shift_id NUMBER(38,0),
    shift_start_time TIME(9),
    shift_end_time TIME(9),
    order_channel VARCHAR(16777216),
    order_ts TIMESTAMP_NTZ(9),
    served_ts VARCHAR(16777216),
    order_currency VARCHAR(3),
    order_amount NUMBER(38,4),
    order_tax_amount VARCHAR(16777216),
    order_discount_amount VARCHAR(16777216),
    order_total NUMBER(38,4)
);

CREATE OR REPLACE TABLE tasty_bytes_all.raw_pos.order_detail
(
    order_detail_id NUMBER(38,0),
    order_id NUMBER(38,0),
    menu_item_id NUMBER(38,0),
    discount_id VARCHAR(16777216),
    line_number NUMBER(38,0),
    quantity NUMBER(5,0),
    unit_price NUMBER(38,4),
    price NUMBER(38,4),
    order_item_discount_amount VARCHAR(16777216)
);

CREATE OR REPLACE TABLE tasty_bytes_all.raw_customer.customer_loyalty
(
    customer_id NUMBER(38,0),
    first_name VARCHAR(16777216),
    last_name VARCHAR(16777216),
    city VARCHAR(16777216),
    country VARCHAR(16777216),
    postal_code VARCHAR(16777216),
    preferred_language VARCHAR(16777216),
    gender VARCHAR(16777216),
    favourite_brand VARCHAR(16777216),
    marital_status VARCHAR(16777216),
    children_count VARCHAR(16777216),
    sign_up_date DATE,
    birthday_date DATE,
    e_mail VARCHAR(16777216),
    phone_number VARCHAR(16777216)
);

In [ ]:
COPY INTO tasty_bytes_all.raw_pos.country
FROM @tasty_bytes_all.public.frostbyte_s3/raw_pos/country/;

COPY INTO tasty_bytes_all.raw_pos.franchise
FROM @tasty_bytes_all.public.frostbyte_s3/raw_pos/franchise/;

COPY INTO tasty_bytes_all.raw_pos.location
FROM @tasty_bytes_all.public.frostbyte_s3/raw_pos/location/;

COPY INTO tasty_bytes_all.raw_pos.menu
FROM @tasty_bytes_all.public.frostbyte_s3/raw_pos/menu/;

COPY INTO tasty_bytes_all.raw_pos.truck
FROM @tasty_bytes_all.public.frostbyte_s3/raw_pos/truck/;

COPY INTO tasty_bytes_all.raw_customer.customer_loyalty
FROM @tasty_bytes_all.public.frostbyte_s3/raw_customer/customer_loyalty/;

COPY INTO tasty_bytes_all.raw_pos.order_header
FROM @tasty_bytes_all.public.frostbyte_s3/raw_pos/order_header/;

COPY INTO tasty_bytes_all.raw_pos.order_detail
FROM @tasty_bytes_all.public.frostbyte_s3/raw_pos/order_detail/;

## 4. トラックのレビュー（`raw_support.truck_reviews`）

出典: Zero to Snowflake v2 の `setup.sql`。ソース: `s3://sfquickstarts/tastybytes-voc/raw_support/truck_reviews/`

`review` 列は多言語の自由記述テキスト（非構造化データ）です。`review_id` の型は v2 の `NUMBER(38,0)` を採用しました（`tb_voc` 系の教材では `NUMBER(18,0)`）。

In [ ]:
CREATE OR REPLACE TABLE tasty_bytes_all.raw_support.truck_reviews
(
    order_id NUMBER(38,0),
    language VARCHAR(16777216),
    source VARCHAR(16777216),
    review VARCHAR(16777216),
    review_id NUMBER(38,0)
);

COPY INTO tasty_bytes_all.raw_support.truck_reviews
FROM @tasty_bytes_all.public.voc_s3/raw_support/truck_reviews/;

## 5. サプライチェーン・POI・事前集計（価格最適化）

出典: `sfguide-price-optimization-using-snowflake-notebooks-and-streamlit/scripts/sql/tb_po_setup.sql`。ソース: `s3://sfquickstarts/frostbyte_tastybytes/{raw_supply_chain,raw_safegraph,harmonized}/`

元の SQL は S3 のフォルダごとにステージを作っていますが、ここでは `frostbyte_s3` ステージのサブパスとして同じファイルを指定しています。ファイル形式は元の指定（`csv_header_ff`）です。

`harmonized.menu_item_aggregate_dt` は名前の末尾が `_dt` ですが、Dynamic Table ではなく通常のテーブルです。

In [ ]:
CREATE OR REPLACE TABLE tasty_bytes_all.raw_supply_chain.item
(
	ITEM_ID NUMBER(38,0),
	NAME VARCHAR(16777216),
	CATEGORY VARCHAR(16777216),
	UNIT VARCHAR(16777216),
	UNIT_PRICE NUMBER(38,9),
	UNIT_CURRENCY VARCHAR(16777216),
	SHELF_LIFE_DAYS NUMBER(38,0),
	VENDOR_ID NUMBER(38,0),
	IMAGE_URL VARCHAR(16777216)
);

CREATE OR REPLACE TABLE tasty_bytes_all.raw_supply_chain.recipe
(
	RECIPE_ID NUMBER(38,0),
	MENU_ITEM_ID NUMBER(38,0),
	MENU_ITEM_LINE_ITEM NUMBER(38,0),
	ITEM_ID NUMBER(38,0),
	UNIT_QUANTITY NUMBER(38,9)
);

CREATE OR REPLACE TABLE tasty_bytes_all.raw_supply_chain.item_prices
(
	ITEM_ID NUMBER(38,0),
	UNIT_PRICE NUMBER(38,2),
	START_DATE DATE,
	END_DATE DATE
);

CREATE OR REPLACE TABLE tasty_bytes_all.raw_supply_chain.price_elasticity
(
	PE_ID NUMBER(11,0),
	MENU_ITEM_ID NUMBER(38,0),
	PRICE NUMBER(38,2),
	CURRENCY VARCHAR(3),
	FROM_DATE DATE,
	THROUGH_DATE DATE,
	DAY_OF_WEEK NUMBER(2,0)
);

CREATE OR REPLACE TABLE tasty_bytes_all.raw_supply_chain.menu_prices
(
	MENU_ITEM_ID NUMBER(38,0),
	SALES_PRICE_USD NUMBER(38,2),
	START_DATE DATE,
	END_DATE DATE
);

CREATE OR REPLACE TABLE tasty_bytes_all.raw_safegraph.core_poi_geometry
(
	PLACEKEY VARCHAR(16777216),
	PARENT_PLACEKEY VARCHAR(16777216),
	SAFEGRAPH_BRAND_IDS VARCHAR(16777216),
	LOCATION_NAME VARCHAR(16777216),
	BRANDS VARCHAR(16777216),
	STORE_ID VARCHAR(16777216),
	TOP_CATEGORY VARCHAR(16777216),
	SUB_CATEGORY VARCHAR(16777216),
	NAICS_CODE NUMBER(38,0),
	LATITUDE FLOAT,
	LONGITUDE FLOAT,
	STREET_ADDRESS VARCHAR(16777216),
	CITY VARCHAR(16777216),
	REGION VARCHAR(16777216),
	POSTAL_CODE VARCHAR(16777216),
	OPEN_HOURS VARIANT,
	CATEGORY_TAGS VARCHAR(16777216),
	OPENED_ON VARCHAR(16777216),
	CLOSED_ON VARCHAR(16777216),
	TRACKING_CLOSED_SINCE VARCHAR(16777216),
	GEOMETRY_TYPE VARCHAR(16777216),
	POLYGON_WKT VARCHAR(16777216),
	POLYGON_CLASS VARCHAR(16777216),
	ENCLOSED BOOLEAN,
	PHONE_NUMBER VARCHAR(16777216),
	IS_SYNTHETIC BOOLEAN,
	INCLUDES_PARKING_LOT BOOLEAN,
	ISO_COUNTRY_CODE VARCHAR(16777216),
	WKT_AREA_SQ_METERS FLOAT,
	COUNTRY VARCHAR(16777216)
);

CREATE OR REPLACE TABLE tasty_bytes_all.harmonized.menu_item_aggregate_dt
(
	DATE DATE,
	DAY_OF_WEEK NUMBER(2,0),
	MENU_TYPE_ID NUMBER(38,0),
	TRUCK_BRAND_NAME VARCHAR(16777216),
	MENU_ITEM_ID NUMBER(38,0),
	MENU_ITEM_NAME VARCHAR(16777216),
	SALE_PRICE NUMBER(38,2),
	BASE_PRICE NUMBER(38,2),
	COST_OF_GOODS_USD NUMBER(38,2),
	COUNT_ORDERS NUMBER(18,0),
	TOTAL_QUANTITY_SOLD NUMBER(17,0),
	COMPETITOR_PRICE VARCHAR(16777216)
);

In [ ]:
COPY INTO tasty_bytes_all.raw_supply_chain.item
FROM @tasty_bytes_all.public.frostbyte_s3/raw_supply_chain/item.csv
file_format = (format_name = 'tasty_bytes_all.public.csv_header_ff');

COPY INTO tasty_bytes_all.raw_supply_chain.recipe
FROM @tasty_bytes_all.public.frostbyte_s3/raw_supply_chain/recipe.csv
file_format = (format_name = 'tasty_bytes_all.public.csv_header_ff');

COPY INTO tasty_bytes_all.raw_supply_chain.item_prices
FROM @tasty_bytes_all.public.frostbyte_s3/raw_supply_chain/item_prices.csv
file_format = (format_name = 'tasty_bytes_all.public.csv_header_ff');

COPY INTO tasty_bytes_all.raw_supply_chain.price_elasticity
FROM @tasty_bytes_all.public.frostbyte_s3/raw_supply_chain/price_elasticity.csv
file_format = (format_name = 'tasty_bytes_all.public.csv_header_ff');

COPY INTO tasty_bytes_all.raw_supply_chain.menu_prices
FROM @tasty_bytes_all.public.frostbyte_s3/raw_supply_chain/menu_prices.csv
file_format = (format_name = 'tasty_bytes_all.public.csv_header_ff');

COPY INTO tasty_bytes_all.raw_safegraph.core_poi_geometry
FROM @tasty_bytes_all.public.frostbyte_s3/raw_safegraph/core_poi_geometry.csv
file_format = (format_name = 'tasty_bytes_all.public.csv_header_ff');

COPY INTO tasty_bytes_all.harmonized.menu_item_aggregate_dt
FROM @tasty_bytes_all.public.frostbyte_s3/harmonized/menu_item_aggregate_dt
file_format = (format_name = 'tasty_bytes_all.public.csv_header_ff');

## 6. 分析用の売上データ

| テーブル | 出典 | ソース |
| --- | --- | --- |
| `analytics.shift_sales` | Snowpark 101 for Data Science（元は `raw.shift_sales`） | `s3://sfquickstarts/frostbyte_tastybytes/analytics/shift_sales/` |
| `analytics.tasty_byte_sales` | ML 予測・異常検知（元は `quickstart.ml_functions.tasty_byte_sales`） | `s3://sfquickstarts/frostbyte_tastybytes/mlpf_quickstart/ml_functions_quickstart.csv` |

In [ ]:
CREATE OR REPLACE TABLE tasty_bytes_all.analytics.shift_sales
(
	location_id NUMBER(19,0),
	city VARCHAR(16777216),
	date DATE,
	shift_sales FLOAT,
	shift VARCHAR(2),
	month NUMBER(2,0),
	day_of_week NUMBER(2,0),
	city_population NUMBER(38,0)
);

COPY INTO tasty_bytes_all.analytics.shift_sales
FROM @tasty_bytes_all.public.frostbyte_s3/analytics/shift_sales/;

CREATE OR REPLACE TABLE tasty_bytes_all.analytics.tasty_byte_sales
(
	DATE DATE,
	PRIMARY_CITY VARCHAR(16777216),
	MENU_ITEM_NAME VARCHAR(16777216),
	TOTAL_SOLD NUMBER(17,0)
);

COPY INTO tasty_bytes_all.analytics.tasty_byte_sales
FROM @tasty_bytes_all.public.frostbyte_s3/mlpf_quickstart/ml_functions_quickstart.csv
file_format = (format_name = 'tasty_bytes_all.public.csv_header_auto_ff');

## 7. アプリ用の注文データ（`app`）

| テーブル | 出典 | ソース |
| --- | --- | --- |
| `app.orders` | Build a Data App（`zero_to_app_db_setup.sql`）。`harmonized.orders_v` と同じ 32 列の非正規化テーブル | `s3://sfquickstarts/frostbyte_tastybytes/app/orders/` |
| `app.app_order_header` / `app.app_order_detail` | React Native アプリ（元は `raw` スキーマ） | `s3://sfquickstarts/frostbyte_tastybytes/app/app_orders/*.csv.gz` |

元の `zero_to_app_db_setup.sql` のステージにはファイル形式の指定がありません。このノートブックでは `csv_ff`（`type = 'csv'` のみ、つまり CSV の既定値）を使っています。

In [ ]:
CREATE OR REPLACE TABLE tasty_bytes_all.app.orders
(
	order_id number(38,0),
	truck_id number(38,0),
	order_ts timestamp_ntz(9),
	order_detail_id number(38,0),
	line_number number(38,0),
	truck_brand_name varchar(16777216),
	menu_type varchar(16777216),
	primary_city varchar(16777216),
	region varchar(16777216),
	country varchar(16777216),
	franchise_flag number(38,0),
	franchise_id number(38,0),
	franchisee_first_name varchar(16777216),
	franchisee_last_name varchar(16777216),
	location_id number(19,0),
	customer_id number(38,0),
	first_name varchar(16777216),
	last_name varchar(16777216),
	e_mail varchar(16777216),
	phone_number varchar(16777216),
	children_count varchar(16777216),
	gender varchar(16777216),
	marital_status varchar(16777216),
	menu_item_id number(38,0),
	menu_item_name varchar(16777216),
	quantity number(5,0),
	unit_price number(38,4),
	price number(38,4),
	order_amount number(38,4),
	order_tax_amount varchar(16777216),
	order_discount_amount varchar(16777216),
	order_total number(38,4)
);

COPY INTO tasty_bytes_all.app.orders
FROM @tasty_bytes_all.public.frostbyte_s3/app/orders/;

CREATE OR REPLACE TABLE tasty_bytes_all.app.app_order_header
(
    order_id NUMBER(19,0),
    month NUMBER(2,0),
    day NUMBER(2,0),
    hour NUMBER(2,0),
    minute NUMBER(2,0),
    second NUMBER(2,0),
    order_total NUMBER(19,3),
    order_tax_amount NUMBER(2,0),
    first_name VARCHAR,
    last_name VARCHAR,
    gender VARCHAR,
    order_status VARCHAR
);

CREATE OR REPLACE TABLE tasty_bytes_all.app.app_order_detail
(
    order_id NUMBER(19,0),
    order_detail_id NUMBER(19,0),
    menu_item_name VARCHAR,
    quantity NUMBER(2,0),
    unit_price NUMBER(19,3)
);

COPY INTO tasty_bytes_all.app.app_order_header
FROM @tasty_bytes_all.public.frostbyte_s3/app/app_orders/app_order_header.csv.gz;

COPY INTO tasty_bytes_all.app.app_order_detail
FROM @tasty_bytes_all.public.frostbyte_s3/app/app_orders/app_order_detail.csv.gz;

## 8. 検査レポートとレビュー分析結果（カスタマーエクスペリエンスアプリ）

出典: `sfguide-tasty-bytes-enhancing-customer-experience/customer-experience-app/setup/setup.sql`。ソース: `s3://sfquickstarts/tastybytes-ece/`

- `raw_doc.inspection_report_raw_extract`: 検査レポート画像から Document AI で抽出済みの結果（`VARIANT`、半構造化データ）
- `raw_doc.inspection_report_ref`: 検査項目の説明とカテゴリ
- `analytics.review_analysis_output`: レビューの分析結果（事前計算済み）
- 内部ステージ `raw_doc.inspection_reports`: 検査レポート画像そのもの（非構造化データ）を `COPY FILES` でコピー

この教材が同じ S3 から読み込む `raw_pos.truck` / `raw_pos.menu` は、セクション 3 の中核テーブルと重複するため読み込みません。

In [ ]:
CREATE OR REPLACE TABLE tasty_bytes_all.raw_doc.inspection_report_raw_extract (
	FILE_NAME VARCHAR(16777216),
	EXTRACTION_TIMESTAMP TIMESTAMP_LTZ(9),
	INSPECTION_REPORT_OBJECT VARIANT
);

CREATE OR REPLACE TABLE tasty_bytes_all.raw_doc.inspection_report_ref (
	IRR_ID NUMBER(38,0),
	COLUMN_NAME VARCHAR(16777216),
	DESCRIPTION VARCHAR(16777216),
	CATEGORY VARCHAR(16777216)
);

CREATE OR REPLACE TABLE tasty_bytes_all.analytics.review_analysis_output (
	TRUCK_ID NUMBER(38,0),
	REVIEW_ID NUMBER(18,0),
	REVIEW VARCHAR(16777216),
	SENTIMENT FLOAT,
	TRANSLATED_REVIEW VARCHAR(16777216),
	RATING VARCHAR(16777216),
	CLEAN_RATING VARCHAR(16777216),
	RECOMMEND VARCHAR(16777216),
	CLEAN_RECOMMEND VARCHAR(16777216),
	CATEGORY_SENTIMENT VARCHAR(16777216)
);

COPY INTO tasty_bytes_all.raw_doc.inspection_report_raw_extract
FROM @tasty_bytes_all.public.ece_s3/raw_doc/inspection_report_raw_extract/;

COPY INTO tasty_bytes_all.raw_doc.inspection_report_ref
FROM @tasty_bytes_all.public.ece_s3/raw_doc/inspection_report_ref/;

COPY INTO tasty_bytes_all.analytics.review_analysis_output
FROM @tasty_bytes_all.public.ece_s3/analytics/review_analysis_output/;

CREATE OR REPLACE STAGE tasty_bytes_all.raw_doc.inspection_reports
COMMENT = 'Inspection reports images'
DIRECTORY = (ENABLE = TRUE)
ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

COPY FILES
  INTO @tasty_bytes_all.raw_doc.inspection_reports
  FROM @tasty_bytes_all.public.ece_s3/raw_doc/inspection_report/;

ALTER STAGE tasty_bytes_all.raw_doc.inspection_reports REFRESH;

## 9. サポートメールと RAG の知識ベース

出典: `sfguide-tasty-bytes-enhancing-customer-experience/customer-support-email-app/setup/setup.sql`。ソース: `s3://sfquickstarts/tastybytes-ece/`

- `raw_support.email_status_app` / `email_response_app`: 受信メールと返信（`VARIANT`）
- `harmonized.documents`: 文書のテキスト（`RELATIVE_PATH`, `RAW_TEXT`）
- `harmonized.chunk_text`: 文書のチャンク
- `harmonized.array_table` → `harmonized.vector_store`: 埋め込みを `ARRAY` で読み込み、`VECTOR(FLOAT, 768)` に変換

RAG チャットボット教材も `s3://sfquickstarts/tastybytes-cx/app/` から同じ列構成の `documents` / `vector_store` を読み込みますが、こちらのソースだけを採用しています。

In [ ]:
CREATE OR REPLACE TABLE tasty_bytes_all.raw_support.email_status_app (
	EMAIL_ID NUMBER(38,0) NOT NULL autoincrement start 1 increment 1 noorder,
	EMAIL_STATUS_CODE NUMBER(38,0),
	EMAIL_STATUS_DEF VARCHAR(255),
	RESPONDED_FLAG BOOLEAN,
	LAST_UPDATED_TS TIMESTAMP_NTZ(9),
	EMAIL_OBJECT VARIANT
);

CREATE OR REPLACE TABLE tasty_bytes_all.raw_support.email_response_app (
	EMAIL_R_ID NUMBER(38,0) NOT NULL autoincrement start 1000 increment 1 noorder,
	EMAIL_ID NUMBER(38,0),
	EMAIL_RESPONSE VARIANT,
	SENT_TS TIMESTAMP_NTZ(9)
);

CREATE OR REPLACE TABLE tasty_bytes_all.harmonized.chunk_text (
	SOURCE VARCHAR(6),
	SOURCE_DESC VARCHAR(16777216),
	FULL_TEXT VARCHAR(16777216),
	SIZE NUMBER(18,0),
	CHUNK VARCHAR(16777216),
	META VARCHAR(16777216),
	INPUT_TEXT VARCHAR(16777216)
);

CREATE OR REPLACE TABLE tasty_bytes_all.harmonized.documents (
	RELATIVE_PATH VARCHAR(16777216),
	RAW_TEXT VARCHAR(16777216)
);

CREATE OR REPLACE TABLE tasty_bytes_all.harmonized.array_table (
	SOURCE VARCHAR(6),
	SOURCE_DESC VARCHAR(16777216),
	FULL_TEXT VARCHAR(16777216),
	SIZE NUMBER(18,0),
	CHUNK VARCHAR(16777216),
	INPUT_TEXT VARCHAR(16777216),
	CHUNK_EMBEDDING ARRAY
);

COPY INTO tasty_bytes_all.raw_support.email_status_app
FROM @tasty_bytes_all.public.ece_s3/raw_support/email_status_app/;

COPY INTO tasty_bytes_all.raw_support.email_response_app
FROM @tasty_bytes_all.public.ece_s3/raw_support/email_response_app/;

COPY INTO tasty_bytes_all.harmonized.documents
FROM @tasty_bytes_all.public.ece_s3/harmonized/documents/;

COPY INTO tasty_bytes_all.harmonized.array_table
FROM @tasty_bytes_all.public.ece_s3/harmonized/vector_store/;

COPY INTO tasty_bytes_all.harmonized.chunk_text
FROM @tasty_bytes_all.public.ece_s3/harmonized/chunk_text/;

CREATE OR REPLACE TABLE tasty_bytes_all.harmonized.vector_store (
	SOURCE VARCHAR(6),
	SOURCE_DESC VARCHAR(16777216),
	FULL_TEXT VARCHAR(16777216),
	SIZE NUMBER(18,0),
	CHUNK VARCHAR(16777216),
	INPUT_TEXT VARCHAR(16777216),
	CHUNK_EMBEDDING VECTOR(FLOAT, 768)
) AS
SELECT
	source,
	source_desc,
	full_text,
	size,
	chunk,
	input_text,
	chunk_embedding::VECTOR(FLOAT, 768)
FROM tasty_bytes_all.harmonized.array_table;

## 10. ファインチューニング用メール（`raw_support.support_emails`）

出典: `sfguide-enhancing-customer-experiences-using-cortex-finetuning/scripts/setup.sql`（元は `CFT_DB.CFT_SCHEMA.SUPPORT_EMAILS`）。ソース: `s3://sfquickstarts/frostbyte_tastybytes/fine_tuning/CFT_QUICKSTART.csv`

In [ ]:
CREATE OR REPLACE TABLE tasty_bytes_all.raw_support.support_emails (
    ID INTEGER,
    TIMESTAMP VARIANT,
    SENDER VARCHAR,
    SUBJECT VARCHAR,
    BODY VARCHAR,
    LABELED_LOCATION VARCHAR,
    LABELED_TRUCK VARCHAR,
    SUPPORT_RESPONSE VARCHAR
);

COPY INTO tasty_bytes_all.raw_support.support_emails
FROM @tasty_bytes_all.public.frostbyte_s3/fine_tuning/CFT_QUICKSTART.csv
FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1);

## 11. 取り込み結果の確認

各テーブルの行数と、検査レポート画像のステージ内のファイル数を表示します。0 行のテーブルがあれば、そのセクションの `COPY INTO` の結果を確認してください。

In [ ]:
SELECT 'raw_pos.country' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_pos.country
UNION ALL SELECT 'raw_pos.franchise' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_pos.franchise
UNION ALL SELECT 'raw_pos.location' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_pos.location
UNION ALL SELECT 'raw_pos.menu' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_pos.menu
UNION ALL SELECT 'raw_pos.truck' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_pos.truck
UNION ALL SELECT 'raw_pos.order_header' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_pos.order_header
UNION ALL SELECT 'raw_pos.order_detail' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_pos.order_detail
UNION ALL SELECT 'raw_customer.customer_loyalty' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_customer.customer_loyalty
UNION ALL SELECT 'raw_support.truck_reviews' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_support.truck_reviews
UNION ALL SELECT 'raw_support.email_status_app' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_support.email_status_app
UNION ALL SELECT 'raw_support.email_response_app' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_support.email_response_app
UNION ALL SELECT 'raw_support.support_emails' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_support.support_emails
UNION ALL SELECT 'raw_supply_chain.item' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_supply_chain.item
UNION ALL SELECT 'raw_supply_chain.recipe' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_supply_chain.recipe
UNION ALL SELECT 'raw_supply_chain.item_prices' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_supply_chain.item_prices
UNION ALL SELECT 'raw_supply_chain.price_elasticity' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_supply_chain.price_elasticity
UNION ALL SELECT 'raw_supply_chain.menu_prices' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_supply_chain.menu_prices
UNION ALL SELECT 'raw_safegraph.core_poi_geometry' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_safegraph.core_poi_geometry
UNION ALL SELECT 'raw_doc.inspection_report_raw_extract' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_doc.inspection_report_raw_extract
UNION ALL SELECT 'raw_doc.inspection_report_ref' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.raw_doc.inspection_report_ref
UNION ALL SELECT 'harmonized.menu_item_aggregate_dt' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.harmonized.menu_item_aggregate_dt
UNION ALL SELECT 'harmonized.documents' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.harmonized.documents
UNION ALL SELECT 'harmonized.chunk_text' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.harmonized.chunk_text
UNION ALL SELECT 'harmonized.array_table' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.harmonized.array_table
UNION ALL SELECT 'harmonized.vector_store' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.harmonized.vector_store
UNION ALL SELECT 'analytics.shift_sales' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.analytics.shift_sales
UNION ALL SELECT 'analytics.tasty_byte_sales' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.analytics.tasty_byte_sales
UNION ALL SELECT 'analytics.review_analysis_output' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.analytics.review_analysis_output
UNION ALL SELECT 'app.orders' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.app.orders
UNION ALL SELECT 'app.app_order_header' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.app.app_order_header
UNION ALL SELECT 'app.app_order_detail' AS table_name, COUNT(*) AS row_count FROM tasty_bytes_all.app.app_order_detail
ORDER BY table_name;

In [ ]:
SELECT COUNT(*) AS inspection_report_files
FROM DIRECTORY(@tasty_bytes_all.raw_doc.inspection_reports);

In [ ]:
ALTER WAREHOUSE tasty_bytes_all_wh SET WAREHOUSE_SIZE = 'xsmall';

## 12. このノートブックで取り込まないデータ

### 採用しなかったソース（同じテーブルの別ソース）

| テーブル | 採用しなかったソース | 使う教材 |
| --- | --- | --- |
| 中核 8 テーブル | `s3://sfquickstarts/tasty-bytes-builder-education/` | Northstar、Modern Data Engineering、宣言型パイプライン |
| 中核 8 テーブル | `s3://sfquickstarts/data-engineering-with-snowpark-python/`（Parquet） | Snowpark Python データエンジニアリング |
| 中核 8 テーブル | `s3://sfquickstarts/sfguide_getting_started_with_running_distributed_pytorch_models_on_snowflake/`（列順・型が一部異なる） | 分散 PyTorch |
| `menu`, `truck`, `order_header` | `s3://sfquickstarts/tastybytes-voc/` | 顧客レビュー分析など `tb_voc` 系 |
| `menu`, `truck` | `s3://sfquickstarts/tastybytes-ece/` | カスタマーエクスペリエンスアプリ |
| `menu` | `s3://sfquickstarts/tastybytes/raw_pos/menu/` | Dynamic Tables 総合ガイドなど |
| `documents`, `vector_store` | `s3://sfquickstarts/tastybytes-cx/app/` | RAG チャットボット |

### S3 から取り込めないデータ

| データ | 理由・入手方法 |
| --- | --- |
| Weather Source、SafeGraph、祝日（Snowflake Public Data） | Marketplace のリスティングを「Get」して共有データベースとして参照する |
| 画像・音声・動画（非構造化データからの洞察） | 教材リポジトリの `data.zip` を内部ステージに手動でアップロードする |
| 四半期別レビューファイル（Iceberg・Tableau） | 利用者の S3 バケットに置き、ストレージ統合を作る必要がある |
| 音声注文の MP3 | 教材の Notebook が外部 API で生成する |
| 価格設定の Excel（`frostbyte_tastybytes/excel/`） | テーブルではなく、Notebook が pandas で読むファイル |
| SSIS 移行の `TastyBytesDB` | SQL Server 用の別スキーマ |